####Staging Layer

#####Reading the CSV file

In [0]:
# Reaing the source path of the dataset
source_path = "s3://clinical-trial4/ctg-studies (4).csv"

In [0]:
# Reading the dataset using spark
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(source_path)

#### Check total count record

In [0]:
# checking the total record count
total_count = df.count()
print("Total Records :", total_count)

#####Calculate chunk Size

In [0]:
# Calculating the chunk size
chunk_size = total_count // 3
print("Chunk Size :", chunk_size)

##### Add row numbers

In [0]:
# Importing required functions for windowing and row numbering
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, monotonically_increasing_id

# Defining a window specification to order rows by a unique increasing ID
window_spec = Window.orderBy(monotonically_increasing_id())

# Adding a row number column to the dataframe based on the window specification
df_with_rownum = df.withColumn(
    "row_num",
    row_number().over(window_spec)
)

####split into 3 dataframes

In [0]:
# Splitting the dataframe into the first chunk based on row numbers
df1 = df_with_rownum.filter(
    df_with_rownum.row_num <= chunk_size
).drop("row_num")

In [0]:
# Splitting the dataframe into the second chunk based on row numbers
df2 = df_with_rownum.filter(
    (df_with_rownum.row_num > chunk_size) &
    (df_with_rownum.row_num <= chunk_size * 2)
).drop("row_num")


In [0]:
# Splitting the dataframe into the third chunk based on row numbers
df3 = df_with_rownum.filter(
    df_with_rownum.row_num > chunk_size * 2
).drop("row_num")

#### Staging paths

In [0]:
# Defining staging paths for each chunk
staging_path_1 = "s3://clinical-trial4/staging/part1/"
staging_path_2 = "s3://clinical-trial4/staging/part2/"
staging_path_3 = "s3://clinical-trial4/staging/part3/"

#### Writing the 3 dataframes

In [0]:
# Writing the first chunk to the staging path as a single CSV file with header
df1.coalesce(1).write.format("csv") \
    .option("header", "true") \
    .mode("overwrite") \
    .save(staging_path_1)

In [0]:
# Writing the second chunk to the staging path as a single CSV file with header
df2.coalesce(1).write.format("csv") \
    .option("header", "true") \
    .mode("overwrite") \
    .save(staging_path_2)

In [0]:
# Writing the third chunk to the staging path as a single CSV file with header
df3.coalesce(1).write.format("csv") \
    .option("header", "true") \
    .mode("overwrite") \
    .save(staging_path_3)